# Compare the evaluation runs

For one version, finish all three LLM runs, then all three scanner runs. Manually fill each results-table.csv before using this notebook. Select one successful run per tool below; do not select runs by their score. Repeat for version B after finishing version A.

Run this notebook from evaluation-runs/. See ../results-notes.md for commands and the table format. Grades and the final percentage are entered manually; this notebook checks them and makes the matrices.


In [ ]:
from pathlib import Path
import json
import math
import re

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd()
if ROOT.name != "evaluation-runs":
    raise ValueError("Start Jupyter from evaluation-runs/ as described in results-notes.md.")
LABELS = ["Found", "Missing", "Partially Found"]
TARGETS = [f"V{i}" for i in range(1, 11)]
LLMS = ("gemini", "groq", "mistral")
SCANNERS = ("semgrep", "codeql", "snyk-code")


## Select the version and runs

Paste only each run folder's name, without the tool's parent path. All six must belong to CASE. Each generated image and count CSV will be saved in the relevant run's confusion-matrices/ folder. An LLM-scanner comparison is generated once and saved in both run folders, with identical row/column orientation. Re-running the same selection replaces its generated matrices; it does not edit original outputs or grading tables.


In [ ]:
CASE = "version-A"  # Change to version-B for the second pass.
RUNS = {
    "gemini": "",
    "groq": "",
    "mistral": "",
    "semgrep": "",
    "codeql": "",
    "snyk-code": "",
}


In [ ]:
def read_answer_key(case):
    if case not in ("version-A", "version-B"):
        raise ValueError("CASE must be version-A or version-B.")
    column = 2 if case == "version-A" else 3
    expected = {}
    for line in (ROOT.parent / "answer-key.md").read_text(encoding="utf-8").splitlines():
        fields = [field.strip() for field in line.strip().strip("|").split("|")]
        if len(fields) == 4 and fields[0] in TARGETS:
            label = fields[column].split(" ")[0]
            if label not in ("Found", "Missing"):
                raise ValueError(f"Unexpected answer-key entry: {line}")
            expected[fields[0]] = label
    if set(expected) != set(TARGETS):
        raise ValueError("The answer key must contain V1 through V10 exactly once.")
    return pd.Series(expected).reindex(TARGETS)


def read_run(tool, name, expected):
    if not name or Path(name).name != name:
        raise ValueError(f"Paste a single run folder name for {tool}.")
    folder = ROOT / tool / name
    if not re.search(rf"_EST_{re.escape(CASE)}(?: \(\d+\))?$", name):
        raise ValueError(f"{tool}: the run folder must be for {CASE}.")
    if tool in LLMS:
        log = json.loads((folder / "run-log.json").read_text(encoding="utf-8"))
        if log.get("case") != CASE or log.get("http_status") != 200 or log.get("error") is not None:
            raise ValueError(f"{tool}: choose a successful, complete run for {CASE}.")
        if not (folder / "raw-output.txt").read_text(encoding="utf-8").strip():
            raise ValueError(f"{tool}: no answer text was saved.")
    else:
        log = (folder / "run-log.txt").read_text(encoding="utf-8")
        if f"Case: {CASE}\n" not in log or "Status: COMPLETED /" not in log:
            raise ValueError(f"{tool}: choose a completed scan for {CASE}.")
    table = pd.read_csv(folder / "results-table.csv", dtype=str, keep_default_na=False)
    columns = ["target_id", "expected_answer", "result", "reported_locations",
               "explanation", "points", "final_accuracy_percentage"]
    if list(table.columns) != columns:
        raise ValueError(f"{tool}: keep the results-table.csv column headers unchanged.")
    if table.target_id.duplicated().any() or set(table.target_id) != {*TARGETS, "TOTAL"}:
        raise ValueError(f"{tool}: the table must have V1 through V10 and one TOTAL row.")
    table = table.set_index("target_id")
    rows = table.loc[TARGETS]
    if not rows.result.isin(LABELS).all():
        raise ValueError(f"{tool}: manually fill every result with Found, Missing, or Partially Found.")
    if not rows.expected_answer.equals(expected.rename("expected_answer")):
        # Series names do not affect equals; target IDs determine the alignment.
        raise ValueError(f"{tool}: expected_answer must match answer-key.md for {CASE}.")
    points = pd.to_numeric(rows.points, errors="raise")
    if not points.isin([0, 0.5, 1]).all():
        raise ValueError(f"{tool}: manually grade each target with 0, 0.5, or 1.")
    total = float(table.loc["TOTAL", "points"])
    percentage = float(table.loc["TOTAL", "final_accuracy_percentage"].rstrip("%"))
    if not math.isclose(total, float(points.sum()), abs_tol=0.001):
        raise ValueError(f"{tool}: TOTAL points must equal the ten target scores.")
    if not math.isclose(percentage, total / 10 * 100, abs_tol=0.05):
        raise ValueError(f"{tool}: final accuracy must equal (TOTAL points / 10) * 100.")
    return folder, rows.result


# Validate all six tables before writing any images.
expected = read_answer_key(CASE)
selected = {tool: read_run(tool, RUNS[tool], expected) for tool in (*LLMS, *SCANNERS)}
print(f"All six {CASE} runs and manual tables are ready.")


## Generate the comparisons

All matrices have the same three categories. Answer-key comparisons put the answer key on rows and the detector on columns. LLM-scanner comparisons put the scanner on rows and the LLM on columns; these show agreement, not correctness against the answer key. Every matrix counts the ten targets once. Partial credit affects the manually entered accuracy score; the matrix keeps Partially Found as its own category.

This makes six detector-versus-answer-key matrices and nine LLM-versus-scanner matrices. Each run gets four associated images and four count CSVs. Titles identify the actual runs used.


In [ ]:
def save_matrix(reference, prediction, reference_name, prediction_name, destinations):
    counts = pd.crosstab(
        pd.Categorical(reference.reindex(TARGETS), categories=LABELS),
        pd.Categorical(prediction.reindex(TARGETS), categories=LABELS),
        dropna=False,
    ).reindex(index=LABELS, columns=LABELS, fill_value=0)
    counts.index.name = reference_name
    counts.columns.name = prediction_name
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(counts, annot=True, fmt="d", cmap="Blues", vmin=0, vmax=10,
                square=True, cbar=False, ax=ax)
    ax.set_xlabel(f"{prediction_name} result")
    ax.set_ylabel(f"{reference_name} result")
    details = [f"{tool}: {RUNS[tool]}" for tool in (reference_name, prediction_name)
               if tool != "answer-key"]
    ax.set_title(f"{CASE}: {prediction_name} vs {reference_name}\n" + "\n".join(details), fontsize=9)
    ax.tick_params(axis="x", rotation=15)
    ax.tick_params(axis="y", rotation=0)
    fig.tight_layout()
    filename = f"{prediction_name}_vs_{reference_name}"
    for folder in destinations:
        destination = folder / "confusion-matrices"
        destination.mkdir(exist_ok=True)
        fig.savefig(destination / f"{filename}.png", dpi=160, bbox_inches="tight")
        counts.to_csv(destination / f"{filename}.csv")
    plt.close(fig)
    return counts


for tool, (folder, results) in selected.items():
    save_matrix(expected, results, "answer-key", tool, [folder])

for llm in LLMS:
    llm_folder, llm_results = selected[llm]
    for scanner in SCANNERS:
        scanner_folder, scanner_results = selected[scanner]
        save_matrix(scanner_results, llm_results, scanner, llm, [llm_folder, scanner_folder])

print(f"Saved 15 distinct comparisons for {CASE}; each run has four PNGs and four count CSVs.")
print("Review the images in each run's confusion-matrices/ folder.")
